In [ ]:
# Run 100% training from scratch as a comparative baseline:
# train_and_evaluate(train_fraction=1, use_foundation=False)

import os
import random
import numpy as np
import scipy.ndimage as ndi
import SimpleITK as sitk
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import (
    MapTransform, Compose, RandSpatialCropd, RandFlipd, ToTensord, AsDiscrete
)
from monai.data import DataLoader, Dataset

# ==========================================
# 1. Strictly aligned custom preprocessing Transform
# ==========================================
class AutoPETPreprocessd(MapTransform):
    def __init__(self, keys):
        super().__init__(keys)
        self.target_spacing_zyx = np.array([3.0, 2.0, 2.0])

    def __call__(self, data):
        d = dict(data)
        
        pet_img = sitk.ReadImage(d['pet_path'])
        ct_img = sitk.ReadImage(d['ct_path'])
        seg_img = sitk.ReadImage(d['seg_path'])
        
        spacing_xyz = np.round(pet_img.GetSpacing()).astype(float)
        orig_spacing_zyx = np.array([spacing_xyz[2], spacing_xyz[1], spacing_xyz[0]])
        
        pet_array = sitk.GetArrayFromImage(pet_img).astype(np.float32)
        ct_array = sitk.GetArrayFromImage(ct_img).astype(np.float32)
        seg_array = sitk.GetArrayFromImage(seg_img).astype(np.uint8) 
        
        # Enforce binarization to prevent anomalous label values
        seg_array = (seg_array > 0).astype(np.uint8)
        
        zoom_factors_pet = orig_spacing_zyx / self.target_spacing_zyx
        pet_resampled = ndi.zoom(pet_array, zoom_factors_pet, order=1)
        zoom_factors_ct = np.array(pet_resampled.shape) / np.array(ct_array.shape)
        ct_resampled = ndi.zoom(ct_array, zoom_factors_ct, order=1)
        
        # Use nearest-neighbor interpolation for labels
        seg_resampled = ndi.zoom(seg_array, zoom_factors_ct, order=0)
        
        body_mask = ct_resampled > -500
        labels, num_features = ndi.label(body_mask)
        
        if num_features > 0:
            counts = np.bincount(labels.ravel())
            counts[0] = 0 
            largest_component = counts.argmax()
            clean_body_mask = (labels == largest_component)
            
            z_idx, y_idx, x_idx = np.where(clean_body_mask)
            pad = 3
            z_min, z_max = max(0, z_idx.min() - pad), min(pet_resampled.shape[0], z_idx.max() + pad + 1)
            y_min, y_max = max(0, y_idx.min() - pad), min(pet_resampled.shape[1], y_idx.max() + pad + 1)
            x_min, x_max = max(0, x_idx.min() - pad), min(pet_resampled.shape[2], x_idx.max() + pad + 1)
            
            bbox = (slice(z_min, z_max), slice(y_min, y_max), slice(x_min, x_max))
            
            pet_cropped = pet_resampled[bbox]
            ct_cropped = ct_resampled[bbox]
            seg_cropped = seg_resampled[bbox]
        else:
            pet_cropped, ct_cropped, seg_cropped = pet_resampled, ct_resampled, seg_resampled
            
        def z_score(vol):
            mu, sigma = np.mean(vol), max(np.std(vol), 1e-8)
            return (vol - mu) / sigma
            
        stacked_img = np.stack([z_score(pet_cropped), z_score(ct_cropped)], axis=0)
        seg_expanded = np.expand_dims(seg_cropped, axis=0)
        
        d["image"] = stacked_img.astype(np.float32)
        d["label"] = seg_expanded.astype(np.float32)
        
        return d

# ==========================================
# 2. Partition strategy with precise balancing and data volume scaling
# ==========================================
def get_segmentation_dataloaders(train_fraction=1.0, batch_size=4):
    base_path = Path('/path/to/your/dataset/AutoPET2025_FDG')
    
    subject_dict = {}
    for pet_path in base_path.rglob('PET.nii.gz'):
        scan_dir = pet_path.parent
        ct_path = scan_dir / 'CT_resample.nii.gz'
        seg_path = scan_dir / 'tumorSeg.nii.gz'
        
        if ct_path.exists() and seg_path.exists():
            subject_id = pet_path.relative_to(base_path).parts[0]
            if subject_id not in subject_dict:
                subject_dict[subject_id] = []
            subject_dict[subject_id].append({
                "pet_path": str(pet_path),
                "ct_path": str(ct_path),
                "seg_path": str(seg_path)
            })
            
    single_scan_subjects = [sub for sub, scans in subject_dict.items() if len(scans) == 1]
    multi_scan_subjects = [sub for sub, scans in subject_dict.items() if len(scans) > 1]
    
    print(f"Parsing successful! Found {len(subject_dict)} valid patients, totaling {sum(len(s) for s in subject_dict.values())} scans.")
    
    random.seed(42)
    random.shuffle(single_scan_subjects)
    random.shuffle(multi_scan_subjects)
    
    test_subjects = single_scan_subjects[:200]
    remaining_singles = single_scan_subjects[200:]
    
    base_train_subjects = []
    train_scan_count = 0
    used_multis = []
    
    for sub in multi_scan_subjects:
        num_scans = len(subject_dict[sub])
        if train_scan_count + num_scans <= 400: 
            base_train_subjects.append(sub)
            train_scan_count += num_scans
            used_multis.append(sub)
            
    needed_singles = 700 - train_scan_count
    base_train_subjects.extend(remaining_singles[:needed_singles])
    
    target_subset_scans = int(700 * train_fraction)
    
    random.seed(1024) 
    random.shuffle(base_train_subjects)
    
    final_train_subjects = []
    final_train_count = 0
    for sub in base_train_subjects:
        if final_train_count >= target_subset_scans:
            break
        final_train_subjects.append(sub)
        final_train_count += len(subject_dict[sub])
    
    test_files = [scan for sub in test_subjects for scan in subject_dict[sub]]
    train_files = [scan for sub in final_train_subjects for scan in subject_dict[sub]]
    
    print(f"=== Experimental Setup: Using {train_fraction*100:.0f}% training data ===")
    print(f"Partition Result -> Train Set: {len(train_files)} scans | Test Set: {len(test_files)} scans")
    
    train_transforms = Compose([
        AutoPETPreprocessd(keys=["image", "label"]),
        RandSpatialCropd(keys=["image", "label"], roi_size=(96, 128, 128), random_size=False),
        ToTensord(keys=["image", "label"])
    ])
    
    test_transforms = Compose([
        AutoPETPreprocessd(keys=["image", "label"]),
        ToTensord(keys=["image", "label"])
    ])
    
    train_ds = Dataset(data=train_files, transform=train_transforms)
    test_ds = Dataset(data=test_files, transform=test_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4) 
    
    return train_loader, test_loader

# ==========================================
# 3. Load Foundation Model weights
# ==========================================
def build_model(device, foundation_ckpt_path="swin_mae_best_v2.pth", use_foundation=True):
    model = SwinUNETR(
        img_size=(96, 128, 128),
        in_channels=2,          
        out_channels=2,         
        feature_size=48,
        depths=(2, 2, 6, 2),
        num_heads=(3, 6, 12, 24),
        norm_name='instance',
        drop_rate=0.0,
        attn_drop_rate=0.0,
        dropout_path_rate=0.05,
        normalize=True,
        use_checkpoint=True,
        spatial_dims=3,
        downsample='mergingv2',
        use_v2=True
    )
    
    # Only load Foundation weights if we are NOT resuming from a fine-tuning checkpoint
    # and the user explicitly asked for foundation initialization
    if use_foundation and foundation_ckpt_path is not None and os.path.exists(foundation_ckpt_path):
        print(f"==> Loading Foundation Model weights: {foundation_ckpt_path}")
        checkpoint = torch.load(foundation_ckpt_path, map_location='cpu', weights_only=True)
        mae_state_dict = checkpoint['model_state_dict']
        
        seg_state_dict = {}
        for k, v in mae_state_dict.items():
            if 'encoder_decoder.' in k:
                new_key = k.split('encoder_decoder.')[1]
                seg_state_dict[new_key] = v
                
        missing_keys, unexpected_keys = model.load_state_dict(seg_state_dict, strict=False)
        print(f"Weight transfer successful! Missing layers: {len(missing_keys)} | Unexpected layers: {len(unexpected_keys)}")
        
    return model.to(device)

# ==========================================
# 4. Fine-tuning main loop with resume training capability
# ==========================================
def train_and_evaluate(train_fraction=1.0, use_foundation=True, total_epochs=100):
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using GPU: {device}")
    
    train_loader, test_loader = get_segmentation_dataloaders(train_fraction=train_fraction, batch_size=4)
    
    pct_str = f"{int(train_fraction*100)}pct"
    model_type = "mae" if use_foundation else "scratch"
    
    # Define filenames for resuming state
    latest_ckpt_path = f"latest_seg_{pct_str}_{model_type}.pth"
    best_ckpt_path = f"best_seg_{pct_str}_{model_type}.pth"

    # Initialize model architecture
    model = build_model(device, foundation_ckpt_path=None, use_foundation=False)
    
    loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scaler = GradScaler('cuda')
    
    # Adjust the total length of the Scheduler to your desired total_epochs (e.g., 100)
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-6)
    
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    
    post_pred = Compose([AsDiscrete(argmax=True, to_onehot=2)])
    post_label = Compose([AsDiscrete(to_onehot=2)])
    
    start_epoch = 0
    best_metric = -1
    val_interval = 20
    
    # ---------------------------------------------------------
    # Resumption Logic
    # ---------------------------------------------------------
    if os.path.exists(latest_ckpt_path):
        print(f"==> Checkpoint file '{latest_ckpt_path}' found, resuming training state...")
        checkpoint = torch.load(latest_ckpt_path, map_location=device, weights_only=False) # Needs False to load optimizer/scheduler
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_metric = checkpoint['best_metric']
        print(f"==> Successfully resumed training from Epoch {start_epoch+1}!")
    else:
        # Only load foundation model weights if this is a completely new training session
        if use_foundation:
            model = build_model(device, foundation_ckpt_path="swin_mae_best_v2.pth", use_foundation=True)
        else:
            print("==> New training session, starting from random weights (Scratch).")
    
    for epoch in range(start_epoch, total_epochs):
        model.train()
        epoch_loss = 0
        current_lr = optimizer.param_groups[0]['lr']
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs} [LR: {current_lr:.2e}]")
        
        for batch_data in pbar:
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast('cuda'):
                outputs = model(inputs)
                loss = loss_function(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
            
        # Update learning rate
        scheduler.step()
        
        print(f"Epoch {epoch+1} Average Loss: {epoch_loss/len(train_loader):.4f}")
        
        # Save the latest state (to allow recovery from future interruptions)
        state_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_metric': best_metric
        }
        torch.save(state_dict, latest_ckpt_path)
        
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                pbar_test = tqdm(test_loader, desc=f"Epoch {epoch+1}/{total_epochs} [Test]")
                for val_data in pbar_test:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    
                    with autocast('cuda'):
                        val_outputs = sliding_window_inference(
                            inputs=val_inputs,
                            roi_size=(96, 128, 128),
                            sw_batch_size=4,
                            predictor=model,
                            overlap=0.5 
                        )
                    
                    val_outputs = [post_pred(i) for i in val_outputs]
                    val_labels = [post_label(i) for i in val_labels]
                    
                    dice_metric(y_pred=val_outputs, y=val_labels)
                    try:
                        hd95_metric(y_pred=val_outputs, y=val_labels)
                    except:
                        pass
                
                mean_dice = dice_metric.aggregate().item()
                try:
                    mean_hd95 = hd95_metric.aggregate().item()
                except:
                    mean_hd95 = float('inf')
                    
                dice_metric.reset()
                hd95_metric.reset()
                
                print(f"👉 Validation - Mean Dice: {mean_dice:.4f} | Mean HD95: {mean_hd95:.4f}")
                
                if mean_dice > best_metric:
                    best_metric = mean_dice
                    # Save the best weights separately (no need to save optimizer state)
                    torch.save(model.state_dict(), best_ckpt_path)
                    print(f"🏆 Saved new best model: {best_ckpt_path} (Dice: {best_metric:.4f})")

if __name__ == '__main__':
    # If latest_seg_10pct_scratch.pth is detected, it will automatically resume
    train_and_evaluate(train_fraction=1, use_foundation=False, total_epochs=100)

In [ ]:
# Run 100% training from MAE as a comparative baseline:
# train_and_evaluate(train_fraction=1, use_foundation=True)

import os
import random
import numpy as np
import scipy.ndimage as ndi
import SimpleITK as sitk
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import (
    MapTransform, Compose, RandSpatialCropd, RandFlipd, ToTensord, AsDiscrete
)
from monai.data import DataLoader, Dataset

# ==========================================
# 1. Strictly aligned custom preprocessing Transform
# ==========================================
class AutoPETPreprocessd(MapTransform):
    def __init__(self, keys):
        super().__init__(keys)
        self.target_spacing_zyx = np.array([3.0, 2.0, 2.0])

    def __call__(self, data):
        d = dict(data)
        
        pet_img = sitk.ReadImage(d['pet_path'])
        ct_img = sitk.ReadImage(d['ct_path'])
        seg_img = sitk.ReadImage(d['seg_path'])
        
        spacing_xyz = np.round(pet_img.GetSpacing()).astype(float)
        orig_spacing_zyx = np.array([spacing_xyz[2], spacing_xyz[1], spacing_xyz[0]])
        
        pet_array = sitk.GetArrayFromImage(pet_img).astype(np.float32)
        ct_array = sitk.GetArrayFromImage(ct_img).astype(np.float32)
        seg_array = sitk.GetArrayFromImage(seg_img).astype(np.uint8) 
        
        # Enforce binarization to prevent anomalous label values
        seg_array = (seg_array > 0).astype(np.uint8)
        
        zoom_factors_pet = orig_spacing_zyx / self.target_spacing_zyx
        pet_resampled = ndi.zoom(pet_array, zoom_factors_pet, order=1)
        zoom_factors_ct = np.array(pet_resampled.shape) / np.array(ct_array.shape)
        ct_resampled = ndi.zoom(ct_array, zoom_factors_ct, order=1)
        
        # Use nearest-neighbor interpolation for labels
        seg_resampled = ndi.zoom(seg_array, zoom_factors_ct, order=0)
        
        body_mask = ct_resampled > -500
        labels, num_features = ndi.label(body_mask)
        
        if num_features > 0:
            counts = np.bincount(labels.ravel())
            counts[0] = 0 
            largest_component = counts.argmax()
            clean_body_mask = (labels == largest_component)
            
            z_idx, y_idx, x_idx = np.where(clean_body_mask)
            pad = 3
            z_min, z_max = max(0, z_idx.min() - pad), min(pet_resampled.shape[0], z_idx.max() + pad + 1)
            y_min, y_max = max(0, y_idx.min() - pad), min(pet_resampled.shape[1], y_idx.max() + pad + 1)
            x_min, x_max = max(0, x_idx.min() - pad), min(pet_resampled.shape[2], x_idx.max() + pad + 1)
            
            bbox = (slice(z_min, z_max), slice(y_min, y_max), slice(x_min, x_max))
            
            pet_cropped = pet_resampled[bbox]
            ct_cropped = ct_resampled[bbox]
            seg_cropped = seg_resampled[bbox]
        else:
            pet_cropped, ct_cropped, seg_cropped = pet_resampled, ct_resampled, seg_resampled
            
        def z_score(vol):
            mu, sigma = np.mean(vol), max(np.std(vol), 1e-8)
            return (vol - mu) / sigma
            
        stacked_img = np.stack([z_score(pet_cropped), z_score(ct_cropped)], axis=0)
        seg_expanded = np.expand_dims(seg_cropped, axis=0)
        
        d["image"] = stacked_img.astype(np.float32)
        d["label"] = seg_expanded.astype(np.float32)
        
        return d

# ==========================================
# 2. Partition strategy with precise balancing and data volume scaling
# ==========================================
def get_segmentation_dataloaders(train_fraction=1.0, batch_size=4):
    base_path = Path('/path/to/your/dataset/AutoPET2025_FDG')
    
    subject_dict = {}
    for pet_path in base_path.rglob('PET.nii.gz'):
        scan_dir = pet_path.parent
        ct_path = scan_dir / 'CT_resample.nii.gz'
        seg_path = scan_dir / 'tumorSeg.nii.gz'
        
        if ct_path.exists() and seg_path.exists():
            subject_id = pet_path.relative_to(base_path).parts[0]
            if subject_id not in subject_dict:
                subject_dict[subject_id] = []
            subject_dict[subject_id].append({
                "pet_path": str(pet_path),
                "ct_path": str(ct_path),
                "seg_path": str(seg_path)
            })
            
    single_scan_subjects = [sub for sub, scans in subject_dict.items() if len(scans) == 1]
    multi_scan_subjects = [sub for sub, scans in subject_dict.items() if len(scans) > 1]
    
    print(f"Parsing successful! Found {len(subject_dict)} valid patients, totaling {sum(len(s) for s in subject_dict.values())} scans.")
    
    random.seed(42)
    random.shuffle(single_scan_subjects)
    random.shuffle(multi_scan_subjects)
    
    test_subjects = single_scan_subjects[:200]
    remaining_singles = single_scan_subjects[200:]
    
    base_train_subjects = []
    train_scan_count = 0
    used_multis = []
    
    for sub in multi_scan_subjects:
        num_scans = len(subject_dict[sub])
        if train_scan_count + num_scans <= 400: 
            base_train_subjects.append(sub)
            train_scan_count += num_scans
            used_multis.append(sub)
            
    needed_singles = 700 - train_scan_count
    base_train_subjects.extend(remaining_singles[:needed_singles])
    
    target_subset_scans = int(700 * train_fraction)
    
    random.seed(1024) 
    random.shuffle(base_train_subjects)
    
    final_train_subjects = []
    final_train_count = 0
    for sub in base_train_subjects:
        if final_train_count >= target_subset_scans:
            break
        final_train_subjects.append(sub)
        final_train_count += len(subject_dict[sub])
    
    test_files = [scan for sub in test_subjects for scan in subject_dict[sub]]
    train_files = [scan for sub in final_train_subjects for scan in subject_dict[sub]]
    
    print(f"=== Experimental Setup: Using {train_fraction*100:.0f}% training data ===")
    print(f"Partition Result -> Train Set: {len(train_files)} scans | Test Set: {len(test_files)} scans")
    
    train_transforms = Compose([
        AutoPETPreprocessd(keys=["image", "label"]),
        RandSpatialCropd(keys=["image", "label"], roi_size=(96, 128, 128), random_size=False),
        ToTensord(keys=["image", "label"])
    ])
    
    test_transforms = Compose([
        AutoPETPreprocessd(keys=["image", "label"]),
        ToTensord(keys=["image", "label"])
    ])
    
    train_ds = Dataset(data=train_files, transform=train_transforms)
    test_ds = Dataset(data=test_files, transform=test_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4) 
    
    return train_loader, test_loader

# ==========================================
# 3. Load Foundation Model weights
# ==========================================
def build_model(device, foundation_ckpt_path="swin_mae_best_v2.pth", use_foundation=True):
    model = SwinUNETR(
        img_size=(96, 128, 128),
        in_channels=2,          
        out_channels=2,         
        feature_size=48,
        depths=(2, 2, 6, 2),
        num_heads=(3, 6, 12, 24),
        norm_name='instance',
        drop_rate=0.0,
        attn_drop_rate=0.0,
        dropout_path_rate=0.05,
        normalize=True,
        use_checkpoint=True,
        spatial_dims=3,
        downsample='mergingv2',
        use_v2=True
    )
    
    # Only load Foundation weights if we are NOT resuming from a fine-tuning checkpoint
    # and the user explicitly asked for foundation initialization
    if use_foundation and foundation_ckpt_path is not None and os.path.exists(foundation_ckpt_path):
        print(f"==> Loading Foundation Model weights: {foundation_ckpt_path}")
        checkpoint = torch.load(foundation_ckpt_path, map_location='cpu', weights_only=True)
        mae_state_dict = checkpoint['model_state_dict']
        
        seg_state_dict = {}
        for k, v in mae_state_dict.items():
            if 'encoder_decoder.' in k:
                new_key = k.split('encoder_decoder.')[1]
                seg_state_dict[new_key] = v
                
        missing_keys, unexpected_keys = model.load_state_dict(seg_state_dict, strict=False)
        print(f"Weight transfer successful! Missing layers: {len(missing_keys)} | Unexpected layers: {len(unexpected_keys)}")
        
    return model.to(device)

# ==========================================
# 4. Fine-tuning main loop with resume training capability (Phantom optimizer bug fixed)
# ==========================================
def train_and_evaluate(train_fraction=1.0, use_foundation=True, total_epochs=100):
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using GPU: {device}")
    
    train_loader, test_loader = get_segmentation_dataloaders(train_fraction=train_fraction, batch_size=4)
    
    pct_str = f"{int(train_fraction*100)}pct"
    model_type = "mae" if use_foundation else "scratch"
    
    # Define filenames for resuming state
    latest_ckpt_path = f"latest_seg_{pct_str}_{model_type}.pth"
    best_ckpt_path = f"best_seg_{pct_str}_{model_type}.pth"

    # ---------------------------------------------------------
    # [Core Fix]: Construct the correct model directly here in one step; do not overwrite it later
    # ---------------------------------------------------------
    foundation_path = "swin_mae_best_v2.pth" if use_foundation else None
    model = build_model(device, foundation_ckpt_path=foundation_path, use_foundation=use_foundation)
    
    # The optimizer is now bound to the unique, absolutely correct model
    loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scaler = GradScaler('cuda')
    
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-6)
    
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    
    post_pred = Compose([AsDiscrete(argmax=True, to_onehot=2)])
    post_label = Compose([AsDiscrete(to_onehot=2)])
    
    start_epoch = 0
    best_metric = -1
    val_interval = 20
    
    # ---------------------------------------------------------
    # Resumption Logic (Only overwrite parameter dictionaries, do not rebuild the model)
    # ---------------------------------------------------------
    if os.path.exists(latest_ckpt_path):
        print(f"==> Checkpoint file '{latest_ckpt_path}' found, resuming training state...")
        checkpoint = torch.load(latest_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_metric = checkpoint['best_metric']
        print(f"==> Successfully resumed training from Epoch {start_epoch+1}!")
    else:
        print("==> No checkpoint found, starting a new training session.")
    
    for epoch in range(start_epoch, total_epochs):
        model.train()
        epoch_loss = 0
        current_lr = optimizer.param_groups[0]['lr']
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs} [LR: {current_lr:.2e}]")
        
        for batch_data in pbar:
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast('cuda'):
                outputs = model(inputs)
                loss = loss_function(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
            
        scheduler.step()
        
        print(f"Epoch {epoch+1} Average Loss: {epoch_loss/len(train_loader):.4f}")
        
        state_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_metric': best_metric
        }
        torch.save(state_dict, latest_ckpt_path)
        
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                pbar_test = tqdm(test_loader, desc=f"Epoch {epoch+1}/{total_epochs} [Test]")
                for val_data in pbar_test:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    
                    with autocast('cuda'):
                        val_outputs = sliding_window_inference(
                            inputs=val_inputs,
                            roi_size=(96, 128, 128),
                            sw_batch_size=4,
                            predictor=model,
                            overlap=0.5 
                        )
                    
                    val_outputs = [post_pred(i) for i in val_outputs]
                    val_labels = [post_label(i) for i in val_labels]
                    
                    dice_metric(y_pred=val_outputs, y=val_labels)
                    try:
                        hd95_metric(y_pred=val_outputs, y=val_labels)
                    except:
                        pass
                
                mean_dice = dice_metric.aggregate().item()
                try:
                    mean_hd95 = hd95_metric.aggregate().item()
                except:
                    mean_hd95 = float('inf')
                    
                dice_metric.reset()
                hd95_metric.reset()
                
                print(f"👉 Validation - Mean Dice: {mean_dice:.4f} | Mean HD95: {mean_hd95:.4f}")
                
                if mean_dice > best_metric:
                    best_metric = mean_dice
                    torch.save(model.state_dict(), best_ckpt_path)
                    print(f"🏆 Saved new best model: {best_ckpt_path} (Dice: {best_metric:.4f})")

if __name__ == '__main__':
    train_and_evaluate(train_fraction=1.0, use_foundation=True, total_epochs=100)